# Simulasi SIR Stokastik pada Graf Erdős–Rényi vs Random Geometric Graph
### Ilustrasi parameter: penyakit mosaik singkong (CMD), penularan antarpetak lewat kutu kebul

**Alur notebook**
1. Setup & parameter
2. Membangkitkan graf ER dan RGG, lalu memverifikasi sifatnya terhadap teori
3. Satu contoh simulasi & kurva epidemi (Monte Carlo)
4. Ukuran akhir wabah vs β: simulasi ER, simulasi RGG, dan rumus teori ER
5. Sweep parameter (analisis sensitivitas): β × ⟨k⟩ × γ
6. Estimasi ambang epidemik numerik & perbandingan dengan rumus ER
7. Visual spasial wabah di RGG

**Model.** Simpul = petak lahan; status S (rentan), I (terinfeksi), R (*removed*: dicabut/dipanen).
Satu langkah = 1 minggu. Tiap langkah, setiap petak I menulari tiap tetangga S dengan peluang β, lalu petak I berpindah ke R dengan peluang γ.
Transmisibilitas per sisi: $T = \dfrac{\beta}{\beta + \gamma - \beta\gamma}$, dan ambang ER: $T\langle k\rangle = 1$.

> File `sir_graf.py` harus berada di folder yang sama dengan notebook ini.

In [ ]:
# --- Khusus Google Colab: ambil kode dari GitHub (ganti USERNAME/NAMA-REPO) ---
# !git clone https://github.com/USERNAME/NAMA-REPO.git
# %cd NAMA-REPO
# !pip install -q -r requirements.txt
import os
os.makedirs("hasil", exist_ok=True)   # semua gambar & CSV disimpan ke folder hasil/
import numpy as np, pandas as pd, networkx as nx, matplotlib.pyplot as plt, math, time
from sir_graf import *

WARNA = {"ER": "#2a78d6", "RGG": "#eb6834", "teori": "#52514e"}
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#b5b4ae", "axes.labelcolor": "#0b0b0b", "xtick.color": "#52514e",
    "ytick.color": "#52514e", "axes.grid": True, "grid.color": "#e8e7e2", "grid.linewidth": 0.8,
    "lines.linewidth": 2, "font.size": 10, "legend.frameon": False,
})
SEED = 2026
rng = np.random.default_rng(SEED)

## 1. Parameter

| Simbol | Arti | Nilai | Sumber |
|---|---|---|---|
| N | jumlah petak | 1000 | asumsi |
| ⟨k⟩ | rata-rata tetangga per petak | 4, 8, 12 | asumsi (kerapatan tanam) |
| roguing | laju cabut tanaman sakit | 0 / 0,03 / 0,1 per hari | Jittamai et al. (2021) |
| panen | laju panen | 0,003 per hari | Jittamai et al. (2021) |
| γ | peluang *removed* per minggu | $1-e^{-(\text{roguing}+\text{panen})\cdot 7}$ | konversi |
| β | peluang tular per sisi per minggu | di-*sweep* 0,005–0,4 | tidak tersedia langsung di literatur |
| I₀ | petak terinfeksi awal | 1% petak | asumsi |

In [ ]:
N = 1000
K_BASE = 8
GAMMA_BASE = gamma_dari_roguing(ROGUING_HARI)          # roguing 0.03/hari + panen
print(f"gamma baseline per minggu = {GAMMA_BASE:.4f}")
print(f"beta kritis teori ER (<k>={K_BASE}) = {beta_kritis_er(K_BASE, GAMMA_BASE):.4f}")

## 2. Membangkitkan graf & verifikasi sifatnya

Yang dicek: rata-rata derajat, variansi derajat (Poisson → variansi ≈ rata-rata), koefisien clustering
(teori: ER ≈ p ≈ ⟨k⟩/N; RGG 2D ≈ $1 - \tfrac{3\sqrt3}{4\pi} \approx 0{,}5865$), dan ukuran komponen terbesar.

In [ ]:
baris = []
for nama in ["ER", "RGG"]:
    for k in [4, 8, 12]:
        A, _ = PEMBANGKIT[nama](N, k, rng)
        G = nx.from_scipy_sparse_array(A)
        d = np.array([x for _, x in G.degree()])
        baris.append({"graf": nama, "k target": k, "<k>": d.mean(), "var(k)": d.var(),
                      "clustering": nx.average_clustering(G),
                      "komponen terbesar (%)": 100 * max(len(c) for c in nx.connected_components(G)) / N})
tabel_graf = pd.DataFrame(baris)
print("Clustering teori RGG 2D =", round(1 - 3 * math.sqrt(3) / (4 * math.pi), 4))
tabel_graf.round(3)

In [ ]:
# Gambar contoh graf kecil (N=200) supaya strukturnya terlihat
fig, ax = plt.subplots(1, 2, figsize=(9, 4.3))
r_small = np.random.default_rng(7)
A_er, _ = graf_er(200, 6, r_small)
A_rg, pos = graf_rgg(200, 6, r_small, torus=False)
for a, (A, judul, p) in zip(ax, [(A_er, "Erdős–Rényi (tanpa jarak)", None),
                                 (A_rg, "Random Geometric Graph (berbasis jarak)", pos)]):
    G = nx.from_scipy_sparse_array(A)
    lay = nx.spring_layout(G, seed=1) if p is None else {i: p[i] for i in range(len(p))}
    nx.draw_networkx_edges(G, lay, ax=a, width=0.5, alpha=0.4, edge_color="#8a8983")
    nx.draw_networkx_nodes(G, lay, ax=a, node_size=14,
                           node_color=WARNA["ER"] if p is None else WARNA["RGG"])
    a.set_title(judul); a.set_axis_off()
plt.tight_layout(); plt.savefig("hasil/fig1_contoh_graf.png", dpi=200, bbox_inches="tight"); plt.show()

In [ ]:
# Distribusi derajat: keduanya Poisson -> perbedaan hasil nanti berasal dari geometri (clustering)
from scipy.stats import poisson
fig, ax = plt.subplots(figsize=(6, 3.6))
kk = np.arange(0, 21)
for nama in ["ER", "RGG"]:
    deg = []
    for _ in range(20):
        A, _ = PEMBANGKIT[nama](N, K_BASE, rng)
        deg.append(np.asarray(A.sum(axis=1)).ravel())
    deg = np.concatenate(deg)
    frek = np.bincount(deg, minlength=21)[:21] / deg.size
    ax.plot(kk, frek, "o-", color=WARNA[nama], ms=5, lw=1.5, label=nama)
ax.plot(kk, poisson.pmf(kk, K_BASE), "--", color=WARNA["teori"], lw=1.5, label=f"Poisson({K_BASE})")
ax.set_xlabel("derajat k"); ax.set_ylabel("P(k)"); ax.set_title("Distribusi derajat (⟨k⟩ = 8)")
ax.legend(); plt.tight_layout(); plt.savefig("hasil/fig2_distribusi_derajat.png", dpi=200, bbox_inches="tight"); plt.show()

## 3. Kurva epidemi (Monte Carlo)
β = 0,08, ⟨k⟩ = 8, γ baseline. Garis = rata-rata, pita = persentil 10–90 dari M ulangan.

In [ ]:
BETA_CONTOH, M_KURVA = 0.08, 200
fig, ax = plt.subplots(figsize=(7, 3.8))
for nama in ["ER", "RGG"]:
    h = monte_carlo(nama, N, K_BASE, BETA_CONTOH, GAMMA_BASE, M_KURVA, rng, simpan_kurva=True)
    L = max(len(c) for c in h["kurva_I"])
    mat = np.array([np.pad(c, (0, L - len(c))) for c in h["kurva_I"]])   # setelah padam, I = 0
    t = np.arange(L)
    ax.fill_between(t, np.percentile(mat, 10, 0), np.percentile(mat, 90, 0), color=WARNA[nama], alpha=0.15, lw=0)
    ax.plot(t, mat.mean(0), color=WARNA[nama], label=f"{nama}  (ukuran akhir {h['ukuran_akhir_rata']:.2f})")
ax.set_xlim(0, 80); ax.set_xlabel("minggu"); ax.set_ylabel("proporsi petak terinfeksi I(t)")
ax.set_title(f"Kurva epidemi, β = {BETA_CONTOH}, ⟨k⟩ = {K_BASE}")
ax.legend(); plt.tight_layout(); plt.savefig("hasil/fig3_kurva_epidemi.png", dpi=200, bbox_inches="tight"); plt.show()

## 4. Ukuran akhir wabah vs β
Titik = simulasi; garis putus-putus = solusi teori ER $s = 1 - e^{-T\langle k\rangle s}$.
Kalau simulasi ER menempel ke garis teori, kode sudah benar. Jarak antara kurva RGG dan teori ER = bias akibat mengabaikan jarak.

In [ ]:
BETA_GRID = np.geomspace(0.001, 0.5, 20)
M_SWEEP = 100          # naikkan ke 200 untuk hasil final skripsi
hasil_b = []
for nama in ["ER", "RGG"]:
    for b in BETA_GRID:
        hasil_b.append(monte_carlo(nama, N, K_BASE, b, GAMMA_BASE, M_SWEEP, rng))
df_b = pd.DataFrame(hasil_b)

fig, ax = plt.subplots(figsize=(7, 4))
bb = np.geomspace(0.001, 0.5, 300)
ax.plot(bb, [ukuran_akhir_teori_er(x, GAMMA_BASE, K_BASE) for x in bb], "--", color=WARNA["teori"], lw=1.5,
        label="teori ER")
for nama in ["ER", "RGG"]:
    d = df_b[df_b.graf == nama]
    ax.errorbar(d.beta, d.ukuran_akhir_rata, yerr=d.ukuran_akhir_sd, fmt="o-", ms=5, lw=1.5, capsize=2,
                color=WARNA[nama], label=f"simulasi {nama}")
ax.axvline(beta_kritis_er(K_BASE, GAMMA_BASE), color=WARNA["teori"], lw=0.8, ls=":")
ax.set_xscale("log"); ax.set_xlabel("β (peluang tular per sisi per minggu)"); ax.set_ylabel("ukuran akhir wabah")
ax.set_title(f"Ukuran akhir wabah, ⟨k⟩ = {K_BASE}, γ = {GAMMA_BASE:.3f}")
ax.legend(); plt.tight_layout(); plt.savefig("hasil/fig4_ukuran_akhir_vs_beta.png", dpi=200, bbox_inches="tight"); plt.show()

## 5. Sweep parameter (analisis sensitivitas)
2 graf × 20 β × 3 ⟨k⟩ × 3 γ = 360 kombinasi, masing-masing M ulangan (graf dibangkitkan ulang tiap ulangan).
Dengan M = 100 butuh sekitar 8–10 menit di laptop biasa. Hasil disimpan ke `hasil_sweep.csv`.

In [ ]:
K_LIST = [4, 8, 12]
ROGUING_LIST = {"tanpa roguing": 0.0, "roguing baseline": 0.03, "roguing intensif": 0.10}
t0, hasil = time.time(), []
for label_g, rog in ROGUING_LIST.items():
    g = gamma_dari_roguing(rog)
    for k in K_LIST:
        for nama in ["ER", "RGG"]:
            for b in BETA_GRID:
                r = monte_carlo(nama, N, k, b, g, M_SWEEP, rng)
                r["skenario_gamma"] = label_g
                hasil.append(r)
df = pd.DataFrame(hasil)
df.to_csv("hasil/hasil_sweep.csv", index=False)
print(f"selesai: {len(df)} kombinasi dalam {time.time() - t0:.0f} detik")
df.head()

In [ ]:
# Heatmap ukuran akhir: baris = <k>, kolom = beta, untuk gamma baseline
from matplotlib.colors import TwoSlopeNorm
d0 = df[df.skenario_gamma == "roguing baseline"]
piv = {n: d0[d0.graf == n].pivot(index="k", columns="beta", values="ukuran_akhir_rata") for n in ["ER", "RGG"]}
selisih = piv["ER"] - piv["RGG"]

fig, ax = plt.subplots(1, 3, figsize=(13, 3.4), sharey=True)
lab_b = [f"{x:.3f}" for x in BETA_GRID]
for a, (judul, M, cmap, norm) in zip(ax, [
        ("ER: ukuran akhir", piv["ER"], "Blues", None),
        ("RGG: ukuran akhir", piv["RGG"], "Blues", None),
        ("Selisih ER − RGG", selisih, "RdBu_r", TwoSlopeNorm(0, -1, 1))]):
    im = a.imshow(M.values, aspect="auto", origin="lower", cmap=cmap, norm=norm,
                  vmin=None if norm else 0, vmax=None if norm else 1)
    a.set_xticks(range(0, len(BETA_GRID), 3)); a.set_xticklabels(lab_b[::3], rotation=45)
    a.set_yticks(range(len(K_LIST))); a.set_yticklabels(K_LIST)
    a.set_title(judul); a.set_xlabel("β"); a.grid(False)
    fig.colorbar(im, ax=a, fraction=0.046)
ax[0].set_ylabel("⟨k⟩")
plt.tight_layout(); plt.savefig("hasil/fig5_heatmap.png", dpi=200, bbox_inches="tight"); plt.show()

## 6. Ambang epidemik: teori ER vs estimasi numerik
Ambang numerik = β terkecil di mana rata-rata ukuran akhir > 10% (interpolasi pada skala log).
Kolom `rasio RGG/ER` > 1 berarti graf spasial butuh β lebih besar agar wabah meluas.

In [ ]:
baris = []
for label_g, rog in ROGUING_LIST.items():
    g = gamma_dari_roguing(rog)
    for k in K_LIST:
        sub = df[(df.skenario_gamma == label_g) & (df.k == k)]
        est = {n: estimasi_beta_kritis(sub[sub.graf == n].beta, sub[sub.graf == n].ukuran_akhir_rata)
               for n in ["ER", "RGG"]}
        baris.append({"skenario": label_g, "gamma": round(g, 4), "<k>": k,
                      "beta_c teori ER": beta_kritis_er(k, g),
                      "beta_c simulasi ER": est["ER"], "beta_c simulasi RGG": est["RGG"],
                      "rasio RGG/ER": est["RGG"] / est["ER"]})
tabel_ambang = pd.DataFrame(baris)
tabel_ambang.to_csv("hasil/tabel_ambang.csv", index=False)
tabel_ambang.round(4)

Catatan membaca tabel:
- Pada ⟨k⟩ = 4, RGG berada di sekitar ambang perkolasi kontinum (⟨k⟩ ≈ 4,5): komponen terbesarnya hanya sekitar 40% petak (lihat tabel di bagian 2), jadi wabah di RGG terbendung oleh struktur graf itu sendiri, bukan hanya oleh β.
- `NaN` berarti ukuran akhir tidak pernah melewati 10% di rentang β yang diuji (atau sudah melewati sejak β terkecil).

## 7. Visual spasial wabah di RGG
Satu realisasi: warna menunjukkan status akhir petak. Terlihat wabah menyebar sebagai "gumpalan" dari titik awal, bukan acak.

In [ ]:
r_vis = np.random.default_rng(11)
A, pos = graf_rgg(N, K_BASE, r_vis)
B_VIS = 0.08
h = simulasi_sir(A, B_VIS, GAMMA_BASE, r_vis, n_awal=3, simpan_status_akhir=True)
fig, ax = plt.subplots(figsize=(5, 5))
warna_status = np.where(h["status"] != S, "#e34948", "#c9c8c2")
ax.scatter(pos[:, 0], pos[:, 1], s=10, c=warna_status, lw=0)
ax.scatter([], [], c="#e34948", s=20, label="pernah terinfeksi (R)")
ax.scatter([], [], c="#c9c8c2", s=20, label="tidak tertular (S)")
ax.set_title(f"RGG, β = {B_VIS}: ukuran akhir {h['ukuran_akhir']:.2f}"); ax.set_aspect("equal")
ax.set_axis_off(); ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=2)
plt.tight_layout(); plt.savefig("hasil/fig6_spasial_rgg.png", dpi=200, bbox_inches="tight"); plt.show()